In [1]:
load_ext jupyter_black

In [44]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.multivariate.multivariate_ols import _MultivariateOLS

In [3]:
df = pd.read_pickle(f"llama_beliefs/Llama-3.1-8B-Instruct_chen_answers.gz")
questions = pd.read_pickle("data/questions.gz")

In [4]:
questions_answers = dict(zip(questions.q_id, questions.correct_answer))

In [45]:
options = {
    "q_50": ["1", "2", "3", "4"],
    "q_51": ["A", "B"],
    "q_52": ["1", "2", "3"],
    "q_53": ["A", "B", "C"],
    "q_54": ["10", "2", "3", "4", "5", "6", "7", "8", "9", "1"],
    "q_55": ["10", "2", "3", "4", "5", "6", "7", "8", "9", "1"],
    "q_56": ["10", "2", "3", "4", "5", "6", "7", "8", "9", "1"],
    "q_57": ["1", "2", "3", "4"],
    "q_58": [
        "1,2",
        "1,3",
        "1,4",
        "2,1",
        "2,3",
        "2,4",
        "3,1",
        "3,2",
        "3,4",
        "4,1",
        "4,2",
        "4,3",
    ],
    "q_59": [
        "Good manners",
        "Independence",
        "Hard work",
        "Feeling of responsibility",
        "Imagination",
        "Tolerance and respect for other people",
        "Thrift, saving money and things",
        "Determination",
        "Religious faith",
        "Not being selfish",
        "Obedience",
    ],
}

id_col = {
    "prism": "conversation_id",
    "chen": "text_id",
    "cad_en": "conversation_id",
    "cad_fr": "conversation_id",
    "cad_pt": "conversation_id",
    "cad_it": "conversation_id",
}

demographics = {
    "prism": [
        "age",
        "gender",
        "employment_status",
        "education",
        "marital_status",
        "english_proficiency",
        "religion",
        "ethnicity",
        "birth_region",
        "reside_region",
        "lm_familiarity",
    ],
    "chen": [
        "label",
        "human_Gender",
    ],
    "cad_en": [
        "annotator_age",
        "annotator_gender",
        "annotator_education_level",
        "annotator_political",
        "annotator_ethnicity",
    ],
    "cad_fr": [
        "annotator_age",
        "annotator_gender",
        "annotator_education_level",
        "annotator_political",
        "annotator_ethnicity",
    ],
    "cad_pt": [
        "annotator_age",
        "annotator_gender",
        "annotator_education_level",
        "annotator_political",
        "annotator_ethnicity",
    ],
    "cad_it": [
        "annotator_age",
        "annotator_gender",
        "annotator_education_level",
        "annotator_political",
        "annotator_ethnicity",
    ],
}

In [16]:
help(sm.multivariate)

Help on module statsmodels.multivariate.api in statsmodels.multivariate:

NAME
    statsmodels.multivariate.api

CLASSES
    builtins.object
        statsmodels.multivariate.factor.FactorResults
        statsmodels.multivariate.pca.PCA
    statsmodels.base.model.Model(builtins.object)
        statsmodels.multivariate.cancorr.CanCorr
        statsmodels.multivariate.factor.Factor
        statsmodels.multivariate.manova.MANOVA

    class CanCorr(statsmodels.base.model.Model)
     |  CanCorr(endog, exog, tolerance=1e-08, missing='none', hasconst=None, **kwargs)
     |
     |  Canonical correlation analysis using singular value decomposition
     |
     |  For matrices exog=x and endog=y, find projections x_cancoef and y_cancoef
     |  such that:
     |
     |      x1 = x * x_cancoef, x1' * x1 is identity matrix
     |      y1 = y * y_cancoef, y1' * y1 is identity matrix
     |
     |  and the correlation between x1 and y1 is maximized.
     |
     |  Attributes
     |  ----------
     | 

In [53]:
for dataset in ["chen", "prism", "cad_en"]:
    df = pd.read_pickle(f"llama_beliefs/Llama-3.1-8B-Instruct_{dataset}_answers.gz")
    questions = pd.read_pickle("data/questions.gz")
    for c in [f"q_{i}" for i in range(50)]:
        df[c] = df[c].str.lower() == questions_answers[c]

    df["accuracy"] = df[[f"q_{i}" for i in range(50)]].mean(axis=1)

    for c in [
        "q_50",
        "q_51",
        "q_52",
        "q_53",
        "q_54",
        "q_55",
        "q_56",
        "q_57",
        "q_58",
        "q_59",
    ]:
        if c == "q_58":
            df[c] = df[c].str.replace(" ", "")
        if c in [
            "q_51",
            "q_53",
            "q_58",
            "q_59",
        ]:
            for v in options[c]:
                df[f"{c}_{v}"] = ~df[c].str.extract("(" + v + ")").isna()
        else:
            df[c] = df[c].str.extract(f"({'|'.join(options[c])})")
            df[c] = df[c].astype(float)
    df = df.drop(columns=[f"q_{i}" for i in range(50)] + ["q_59", "q_60"])

    df_beliefs = pd.read_pickle(
        f"llama_beliefs/Llama-3.1-8B-Instruct_{dataset}_beliefs_preprocessed.gz"
    )
    cols = [
        c
        for c in df_beliefs.columns
        if "shared_extracted_" in c or "value_JSON_" in c or "unknown_token_" in c
    ] + ["revealed_Gender"]
    if "human_Gender" in df_beliefs.columns:
        cols += ["human_Gender"]
    demographics[dataset] += cols
    cols.append(id_col[dataset])
    df = df.merge(df_beliefs[cols], on=id_col[dataset])

    # df_linguistic = pd.read_pickle(f"data/{dataset + '_utterances' if dataset != 'chen' else dataset}_linguistic.gz")
    # for c in ["politeness_user_prompt", "politeness_model_response"]:
    #     if c in df_linguistic:
    #         df_linguistic[c] = df_linguistic[c].replace(
    #             {"impolite": -2, "neutral": 0, "polite": 2, "somewhat polite": 1}
    #         )
    # df_linguistic = df_linguistic.rename(columns={"gpt_description": "topic"})
    # if dataset != "chen":
    #     if dataset == 'prism':
    #         demographics[dataset] += ["model_name"]
    #     demographics[dataset] += ['topic']
    #     group_cols = [id_col[dataset]] + demographics[dataset]
    #     df_linguistic = (
    #     df_linguistic.groupby( group_cols)[
    #         [c for c in df.columns if "_model_response" in c or "_user_prompt" in c]
    #     ]
    #     .mean()
    #     .reset_index()
    #     )
    # df = df.merge(df_linguistic[[id_col[dataset]]+[c for c in df.columns if "_model_response" in c or "_user_prompt" in c]], on=id_col[dataset])
    # demographics[dataset] += [c for c in df.columns if "_model_response" in c or "_user_prompt" in c]

    for col in (
        [
            "accuracy",
            "q_50",
            "q_52",
            "q_54",
            "q_55",
            "q_56",
            "q_57",
        ]
        + [f"q_51_{v}" for v in options["q_51"] if v in df]
        + [f"q_53_{v}" for v in options["q_53"] if v in df]
        + [f"q_58_{v}" for v in options["q_58"] if v in df]
        + [f"q_59_{v}" for v in options["q_59"] if v in df]
    ):
        filtered_df = df.loc[~df[col].isna()]
        X = pd.get_dummies(
            filtered_df[demographics[dataset]], dtype=float, drop_first=True
        )
        X = sm.add_constant(X)
        if col in [
            "q_51",
            "q_53",
            "q_58",
            "q_59",
        ]:
            y = pd.get_dummies(filtered_df[col], dtype=float)
            model11 = _MultivariateOLS(y, X).fit()
            print(model11.summary())
        else:
            y = filtered_df[col]
            model11 = sm.OLS(y, X).fit()
            print(model11.summary())

0      0.78
1      0.74
2      0.82
3      0.78
4      0.80
       ... 
505    0.78
506    0.80
507    0.80
508    0.80
509    0.74
Name: accuracy, Length: 510, dtype: float64
                            OLS Regression Results                            
Dep. Variable:               accuracy   R-squared:                       0.096
Model:                            OLS   Adj. R-squared:                  0.008
Method:                 Least Squares   F-statistic:                     1.097
Date:                Wed, 18 Feb 2026   Prob (F-statistic):              0.314
Time:                        18:04:17   Log-Likelihood:                 1033.8
No. Observations:                 510   AIC:                            -1976.
Df Residuals:                     464   BIC:                            -1781.
Df Model:                          45                                         
Covariance Type:            nonrobust                                         
                                  

/home/vera/miniconda3/envs/usermodel/lib/python3.12/site-packages/statsmodels/stats/stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 6 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "
/home/vera/miniconda3/envs/usermodel/lib/python3.12/site-packages/statsmodels/regression/linear_model.py:1795: RuntimeWarning: divide by zero encountered in divide
  return 1 - (np.divide(self.nobs - self.k_constant, self.df_resid)
/home/vera/miniconda3/envs/usermodel/lib/python3.12/site-packages/statsmodels/regression/linear_model.py:1795: RuntimeWarning: invalid value encountered in scalar multiply
  return 1 - (np.divide(self.nobs - self.k_constant, self.df_resid)
/home/vera/miniconda3/envs/usermodel/lib/python3.12/site-packages/statsmodels/regression/linear_model.py:1717: RuntimeWarning: divide by zero encountered in scalar divide
  return np.dot(wresid, wresid) / self.df_resid


                            OLS Regression Results                            
Dep. Variable:                   q_57   R-squared:                       0.131
Model:                            OLS   Adj. R-squared:                  0.039
Method:                 Least Squares   F-statistic:                     1.418
Date:                Wed, 18 Feb 2026   Prob (F-statistic):             0.0471
Time:                        18:04:18   Log-Likelihood:                -237.05
No. Observations:                 448   AIC:                             562.1
Df Residuals:                     404   BIC:                             742.7
Df Model:                          43                                         
Covariance Type:            nonrobust                                         
                                                              coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------